# Incremental Learning

In [3]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import uuid

### Giả lập tạo embedding (thay thế bằng real embedding từ face_recognition trong thực tế)

In [4]:
def generate_fake_embedding(seed):
    np.random.seed(seed)
    return np.random.rand(1, 128)

### Bộ nhớ: lưu danh sách {id, embedding, số lần gặp}
### Ngưỡng nhận dạng (cosine similarity)

In [7]:
memory = []
SIMILARITY_THRESHOLD = 0.75

## Hàm nhận diện hoặc thêm người mới

In [8]:
def recognize_or_add_face(new_embedding):
    global memory

    if len(memory) == 0:
        new_id = str(uuid.uuid4())[:8]
        memory.append({"id": new_id, "embedding": new_embedding, "seen": 1})
        return f"New person detected → Assigned ID: {new_id}"

    # Tính cosine similarity với tất cả người trong bộ nhớ
    sims = [cosine_similarity(new_embedding, person["embedding"])[0][0] for person in memory]
    best_idx = int(np.argmax(sims))
    best_score = sims[best_idx]

    if best_score > SIMILARITY_THRESHOLD:
        # Cập nhật thông tin người cũ
        person = memory[best_idx]
        person["embedding"] = (person["embedding"] * person["seen"] + new_embedding) / (person["seen"] + 1)
        person["seen"] += 1
        return f"Recognized as ID: {person['id']} (confidence: {best_score:.2f})"
    else:
        # Thêm người mới
        new_id = str(uuid.uuid4())[:8]
        memory.append({"id": new_id, "embedding": new_embedding, "seen": 1})
        return f"New person detected → Assigned ID: {new_id}"

## Demo mô phỏng: robot gặp các khuôn mặt khác nhau

In [9]:
def run_demo():
    logs = []
    logs.append(recognize_or_add_face(generate_fake_embedding(seed=1)))  # Người A
    logs.append(recognize_or_add_face(generate_fake_embedding(seed=2)))  # Người B
    logs.append(recognize_or_add_face(generate_fake_embedding(seed=1)))  # Người A lần nữa
    logs.append(recognize_or_add_face(generate_fake_embedding(seed=3)))  # Người C
    logs.append(recognize_or_add_face(generate_fake_embedding(seed=2)))  # Người B lần nữa
    logs.append(recognize_or_add_face(generate_fake_embedding(seed=4)))  # Người D

    logs.append("\nMemory State:")
    logs.extend([f"ID: {m['id']}, seen: {m['seen']}" for m in memory])

    return "\n".join(logs)

## Demo

In [18]:

if __name__ == "__main__":
    print(run_demo())

Recognized as ID: 3383190b (confidence: 0.93)
Recognized as ID: 3383190b (confidence: 0.92)
Recognized as ID: 3383190b (confidence: 0.93)
Recognized as ID: 3383190b (confidence: 0.88)
Recognized as ID: 3383190b (confidence: 0.92)
Recognized as ID: 3383190b (confidence: 0.87)

Memory State:
ID: 3383190b, seen: 54
